In [1]:
import logging
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, SummarizationMiddleware
from langchain_core.runnables import RunnableConfig

In [1]:
from optclaw.config import get_app_config
from optclaw.models import create_chat_model
from langchain.agents import create_agent

In [ ]:
model = create_chat_model(thinking_enabled=False)

Creating model 'doubao-seed-1.8' with settings: {'model': 'doubao-seed-1-8-251228', 'timeout': 600.0, 'max_retries': 3, 'api_base': 'https://ark.cn-beijing.volces.com/api/v3', 'api_key': '31d61644-628b-4332-9e5c-1987c99f3427', 'extra_body': {'thinking': {'type': 'disabled'}}, 'reasoning_effort': 'minimal'} and kwargs: {}
Model class: optclaw.models.patched_deepseek:PatchedChatDeepSeek


In [ ]:
# 创建Agent
agent = create_agent(
    model=model,
    system_prompt="你好"
)

# 正确调用
response = agent.invoke({
    "messages": [{"role": "user", "content": "今天天气怎么样？"}]
})

print(response)

{'messages': [HumanMessage(content='今天天气怎么样？', additional_kwargs={}, response_metadata={}, id='73494b05-c5b9-4c99-b61e-f1b6222b665b'), AIMessage(content='你好呀！天气情况会因地区不同而有所差异，你可以告诉我你所在的具体城市，我来帮你查询当地的天气哦~ 另外，也可以通过手机自带的天气APP或者气象网站，实时了解更准确的天气信息呢！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 38, 'total_tokens': 95, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'deepseek', 'model_name': 'doubao-seed-1-8-251228', 'system_fingerprint': None, 'id': '021777363837175515075f0c150adeda95477c237460c2169f051', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dd323-e8c0-7251-b69f-1a22024aae65-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 57, 'total_tokens': 95,

In [ ]:
# from typing_extensions import override
# from typing import override

In [5]:
from optclaw.agents.middlewares.thread_data_middleware import ThreadDataMiddleware
c = ThreadDataMiddleware()

In [ ]:
# Agent 的实例化不是单一位置，而是一个分层架构。让我为你梳理完整的实例化链条。

# ## Agent 实例化的完整路径

# ### 1. 最底层：LangChain 原语
# `langchain.agents.create_agent` 是真正的底层创建函数，它返回一个 `CompiledStateGraph`（编译后的状态图）。DeerFlow 的所有工厂都最终调用它。

# ### 2. SDK 层：纯参数工厂
# [create_deerflow_agent](backend/packages/harness/deerflow/agents/factory.py#L61-L154)  
# 这是 SDK 级别的入口点。它接受纯 Python 参数（模型、工具、中间件、系统提示等），不读取任何配置文件。如果你想在代码中直接创建一个 agent 而不依赖配置系统，就用它。

# ```python
# # SDK 用法示例
# agent = create_deerflow_agent(
#     model=ChatOpenAI(model="gpt-4"),
#     tools=[my_tool],
#     system_prompt="You are a helpful assistant",
# )
# ```

# ### 3. 应用层：配置驱动工厂
# [make_lead_agent](backend/packages/harness/deerflow/agents/lead_agent/agent.py#L280-L358)  
# 这是生产环境的入口。它从配置文件读取设置（模型名称、thinking 模式、是否启用子 agent、工具组等），然后调用 `create_agent`。整个系统启动时，实际走的是这条路径。

# ### 4. 运行时：执行入口
# [backend/packages/harness/deerflow/runtime/runs/](backend/packages/harness/deerflow/runtime/runs/) 中的 `manager.py` 和 `worker.py` 负责实际的 agent 实例调用和执行流管理。这里不是创建 agent，而是把已经创建好的 agent 图实例化运行。

# ## 总结
# - **如果你在开发 SDK** 或写单元测试：直接用 [create_deerflow_agent](backend/packages/harness/deerflow/agents/factory.py#L61-L154)  
# - **如果你在部署生产环境**：实例化通过 [make_lead_agent](backend/packages/harness/deerflow/agents/lead_agent/agent.py#L280-L358) 完成，由配置驱动  
# - **底层真相**：最终都是 LangChain 的 `create_agent` 函数创建状态图

# 如果你需要了解运行时如何调度这些 agent，可以查看 [运行时管理](backend/packages/harness/deerflow/runtime/runs/manager.py)。

# [主导 Agent 构建](10-lead-agent-construction)  
# [中间件系统设计](12-middleware-system-design)  
# [架构概述](8-architecture-overview)

In [ ]:
# #!/usr/bin/env python
# """
# Debug script for lead_agent.
# Run this file directly in VS Code with breakpoints.

# Requirements:
#     Run with `uv run` from the backend/ directory so that the uv workspace
#     resolves deerflow-harness and app packages correctly:

#         cd backend && PYTHONPATH=. uv run python debug.py

# Usage:
#     1. Set breakpoints in agent.py or other files
#     2. Press F5 or use "Run and Debug" panel
#     3. Input messages in the terminal to interact with the agent
# """

# import asyncio
# import logging

# from dotenv import load_dotenv
# from langchain_core.messages import HumanMessage

# from deerflow.agents import make_lead_agent

# load_dotenv()

# logging.basicConfig(
#     level=logging.INFO,
#     format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
#     datefmt="%Y-%m-%d %H:%M:%S",
# )


# async def main():
#     # Initialize MCP tools at startup
#     try:
#         from deerflow.mcp import initialize_mcp_tools

#         await initialize_mcp_tools()
#     except Exception as e:
#         print(f"Warning: Failed to initialize MCP tools: {e}")

#     # Create agent with default config
#     config = {
#         "configurable": {
#             "thread_id": "debug-thread-001",
#             "thinking_enabled": True,
#             "is_plan_mode": True,
#             # Uncomment to use a specific model
#             "model_name": "kimi-k2.5",
#         }
#     }

#     agent = make_lead_agent(config)

#     print("=" * 50)
#     print("Lead Agent Debug Mode")
#     print("Type 'quit' or 'exit' to stop")
#     print("=" * 50)

#     while True:
#         try:
#             user_input = input("\nYou: ").strip()
#             if not user_input:
#                 continue
#             if user_input.lower() in ("quit", "exit"):
#                 print("Goodbye!")
#                 break

#             # Invoke the agent
#             state = {"messages": [HumanMessage(content=user_input)]}
#             result = await agent.ainvoke(state, config=config, context={"thread_id": "debug-thread-001"})

#             # Print the response
#             if result.get("messages"):
#                 last_message = result["messages"][-1]
#                 print(f"\nAgent: {last_message.content}")

#         except KeyboardInterrupt:
#             print("\nInterrupted. Goodbye!")
#             break
#         except Exception as e:
#             print(f"\nError: {e}")
#             import traceback

#             traceback.print_exc()


# if __name__ == "__main__":
#     asyncio.run(main())


In [6]:
from typing import TypedDict

class User(TypedDict):
    name: str
    age: int